# Netflix Dataset

**Autor** Adi Zebic

**Datum** 29.05.2026

U ovom radu cu analizirati Netflix kao uzorak trzista striminga kako bih pruzio uvide za stratesku odluku o ulasku na trziste.

In [18]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ast
import warnings
warnings.filterwarnings("ignore")

In [19]:
df = pd.read_csv("finalni_netflix_data.csv")

print("Broj redova i kolona:", df.shape)
print("\nKolone:", df.columns.tolist())
print("\nTipovi podataka:")
print(df.dtypes)

df.head()

Broj redova i kolona: (5850, 15)

Kolone: ['id', 'title', 'type', 'description', 'release_year', 'age_certification', 'runtime', 'genres', 'production_countries', 'seasons', 'imdb_id', 'imdb_score', 'imdb_votes', 'tmdb_popularity', 'tmdb_score']

Tipovi podataka:
id                          str
title                       str
type                        str
description                 str
release_year              int64
age_certification           str
runtime                   int64
genres                      str
production_countries        str
seasons                 float64
imdb_id                     str
imdb_score              float64
imdb_votes              float64
tmdb_popularity         float64
tmdb_score              float64
dtype: object


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


In [20]:
df = df.dropna(subset=["release_year","type"])
df["imdb_score"]=df["imdb_score"].fillna(0)
df['tmdb_score']=df['tmdb_score'].fillna(0)
df['tmdb_popularity']=df['tmdb_popularity'].fillna(0)

print(df.isnull().sum())

id                         0
title                      1
type                       0
description               18
release_year               0
age_certification       2619
runtime                    0
genres                     0
production_countries       0
seasons                 3744
imdb_id                  403
imdb_score                 0
imdb_votes               498
tmdb_popularity            0
tmdb_score                 0
dtype: int64


## Trendovi kroz vrijeme

Prikazat cu kako se broj filmova i serija mijenjao kroz godine.

In [21]:
trendovi = df.groupby(["release_year","type"]).size().reset_index()
trendovi.columns = ["Godina","Tip","Broj"]
trendovi = trendovi[trendovi["Godina"]>=2000]

fig1 = px.line(
    trendovi,
    x="Godina",
    y="Broj",
    color="Tip",
    title="Broj filmova i serija po godini",
    labels={"Godina":"Godina","Broj":"Broj naslova","Tip":"Tip sadrzaja"},
    markers=True
)
fig1.show()

### Zakljucak

Broj sadrzaja na Netflixu znacajno raste od 2015 godine,sa posebnim ubrzanjem od 2018. i 2021.Filmovi su zastupljeniji od serija,ali serije biljeze brzi rast.

## Struktura sadrzaja
Prikazat cu koliki je udio filmova u odnosu na serije na platformi.

In [22]:
udio = df["type"].value_counts().reset_index()
udio.columns = ["Tip","Broj"]
fig2 = px.pie(
    udio,
    names="Tip",
    values="Broj",
    title="Udio filmova i serija na Netflixu",
    hole =0.3,
    color_discrete_sequence=["red","blue"]
)

fig2.update_traces(textposition="inside", textinfo="percent+label")
fig2.show()

## Trajanje filmova

Prikazat cu koliko filmova spada u koje trajanje,koji je format najzastupljeniji.

In [23]:
filmovi = df[df["type"]=="MOVIE"]

fig3 = px.histogram(
    filmovi,
    x="runtime",
    nbins=40,
    title="Raspodjela trajanja filmova",
    labels={"runtime":"Trajanje (min)","count":"Broj filmova"},
    color_discrete_sequence=["green"]
)

fig3.update_layout(bargap=0.1)
fig3.show()

### Zakljucak 

Vecina filmova na netflixu traje izmedju 80 i 110 minuta,postoji manji broj veoma dugih filmova koji traju preko 150 minuta.

## Top 10 zanrova
Prikazat cu koji su to zanrovi najzastupjeniji na platformi.

In [24]:
svi_zanrovi = []
for red in df["genres"].dropna():
    try:
        lista = ast.literal_eval(red)
        svi_zanrovi.extend(lista)
    except:
        pass

zanrovi_df = pd.Series(svi_zanrovi).value_counts().head(10).reset_index()
zanrovi_df.columns = ["Zanr","Broj"]

fig4 = px.bar(
    zanrovi_df,
    x="Zanr",
    y="Broj",
    title="Top 10 zanrova na Netflixu",
    labels={"Zanr":"Zanr","Broj":"Broj naslova"},
    color="Zanr",
    color_continuous_scale="Reds"
)

fig4.show()

## Ocjne korisnika

Prikazat cu kako su ocijenjeni filmovi i serije na oba sistema ocjenjivanja.

In [25]:
df_ocjene =df[df["imdb_score"]>0]
fig5 = px.box(
    df_ocjene,
    x="type",
    y="imdb_score",
    title="IMDb ocjene: filmovi vs serije",
    labels={"type":"Tip sadrzaja","imdb_score":"IMDb ocjena"},
    color_discrete_sequence=["orange","purple"]
)

fig5.show()

In [26]:
df_tmdb = df[df["tmdb_score"]>0]

fig6 = px.box(
    df_tmdb,
    x="type",
    y="tmdb_score",
    color="type",
    title="TMDB ocjene: filmovi vs serije",
    labels={"type":"Tip sadrzaja","tmdb_score":"TMDB ocjena"},
    color_discrete_sequence=["cyan","magenta"]
)
fig6.show()

## Odnos ocjena i popularnosti

Prikazat cu da li bolji TMDB score garantuje vecu popularnost.

In [27]:
df_scatter = df[(df["tmdb_score"]>0) & (df["tmdb_popularity"]<500)]

fig7 = px.scatter(
    df_scatter,
    x="tmdb_score",
    y="tmdb_popularity",
    color="type",
    hover_name="title",
    title="TMDB ocjena vs popularnost",
    labels ={
        "tmdb_score":"TMDB ocjena",
        "tmdb_popularity":"TMDB popularnost",
        "type":"Tip sadrzaja"
    },
    trendline="ols",
    color_discrete_sequence=["teal","salmon"]
)
fig7.show()

## Zaklucci i poslovne preporuke

=Netflix je dominatno fokusiran na filmove,ali serije brze rastu i psotaju sve vazniji segment platforme.

-Sadrzaj se ubrzano povecava od 2015. godine.

=Drama,komedija i dokumentarci su najzastupljeniji zanrovi.

=Serije generalno imaju nesto vise IMDB ocjene od fimova.

=Veza izmedju TMDB ocjene i poularnosti postoji,ali nije linearna.

### Preporuke za menadzment

-Ulaganje u serijeske formate pokazuje se kao perspektivna strategija jer taj segment biljezi stabilan rast.

-Drama i komedija su najsigurniji zanrovi za pocetak.

-Partnerstvo sa postojecim platformama moze biti brzi i jeftiniji ulazak na trziste u poredjenju sa izgradjnom sopstvene platforme od nule.

In [28]:
html_sadrzaj = " "
for fig in [fig1,fig2,fig3,fig4,fig5,fig6,fig7]:
    html_sadrzaj += pio.to_html(fig, full_html=False, include_plotlyjs='cdn')
    
html_template = f"""
<!DOCTYPE html>
<html lang="sr">
<head>
    <meta charset="UTF-8">
    <title>Netflix Analiza</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #f0f0f0;
            color: #333;
            padding: 30px;
        }}
        h1 {{
            color: #e50914;
        }}
        p {{
            color: #555;
        }}
        .grafikon {{
            background: #fff;
            padding: 20px;
            margin-bottom: 30px;
            border-radius: 8px;
        }}
    </style>
</head>
<body>
    <h1>Analiza Netflix kataloga</h1>
    <p>Interaktivni izvjestaj za menadzment.Koristite hover i zoom opcije za detaljniju analizu.</p>
    <div class="grafikon">{html_sadrzaj}</div>
</body>
</html>
"""

with open("Adi_Zebic_Finalni_Izvjestaj.html","w", encoding="utf-8") as f:
    f.write(html_template)
print("HTML fajl uspjesno saccuvan")

HTML fajl uspjesno saccuvan
